In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("Dataset/PSCompPars_2025.11.07_05.13.55.csv")

Fetching only required dataframe

In [3]:
print("--------------------------------------------------------------------------")
print(">> Fetching required dataframe...")

initial_df_columns = [
    'pl_name',
    'hostname',
    'pl_bmasse',
    'pl_rade',
    'pl_orbsmax',
    'pl_orbeccen',
    'pl_eqt',
    'st_teff',
    'st_mass',
    'sy_pnum'
]

initial_df = df[initial_df_columns].copy()
print(">> Data fetched successfully")
print("--------------------------------------------------------------------------")

print(">> Preview of raw dataframe:")
print(initial_df.head())
print("--------------------------------------------------------------------------")

print(">> Renaming column names...")

rename_columns = {
    'pl_name': 'planet_name',
    'hostname': 'star_name',
    'pl_bmasse': 'planet_mass_earth',
    'pl_rade': 'planet_radius_earth',
    'pl_orbsmax': 'planet_orb_distance',
    'pl_orbeccen': 'planet_orb_eccen',
    'pl_eqt': 'temp_kelvin',
    'st_teff': 'star_temp',
    'st_mass': 'star_mass',
    'sy_pnum': 'system_planet_count'
}

initial_df.rename(columns=rename_columns, inplace=True)
print(">> Column renaming completed")
print("--------------------------------------------------------------------------")

print(">> Preview after renaming:")
print(initial_df.head())
print("--------------------------------------------------------------------------")

--------------------------------------------------------------------------
>> Fetching required dataframe...
>> Data fetched successfully
--------------------------------------------------------------------------
>> Preview of raw dataframe:
      pl_name  hostname    pl_bmasse  pl_rade  pl_orbsmax  pl_orbeccen  \
0    11 Com b    11 Com  4914.898486     12.2       1.178        0.238   
1    11 UMi b    11 UMi  4684.814200     12.3       1.530        0.080   
2    14 And b    14 And  1131.151301     13.1       0.775        0.000   
3    14 Her b    14 Her  2559.472162     12.6       2.774        0.373   
4  16 Cyg B b  16 Cyg B   565.737400     13.5       1.660        0.680   

   pl_eqt  st_teff  st_mass  sy_pnum  
0     NaN   4874.0     2.09        1  
1     NaN   4213.0     2.78        1  
2     NaN   4888.0     1.78        1  
3     NaN   5338.0     0.91        2  
4     NaN   5750.0     1.08        1  
--------------------------------------------------------------------------
>> R

In [4]:
print("--------------------------------------------------------------------------")
print(">> Dataset overview (summary statistics)")

display(initial_df.describe().T)

print("--------------------------------------------------------------------------")
print(">> Missing values per column")
print(initial_df.isnull().sum())
print("--------------------------------------------------------------------------")


--------------------------------------------------------------------------
>> Dataset overview (summary statistics)


,count,mean,std,min,25%,50%,75%,max
planet_mass_earth,6011.0,388.815775,1111.691567,0.0200,4.130000,9.1000,181.798303,12651.5000
planet_radius_earth,6018.0,5.809140,5.345461,0.3098,1.820000,2.8300,11.957201,77.3421
planet_orb_distance,5738.0,15.628634,351.737777,0.0044,0.052323,0.1016,0.300675,19000.0000
planet_orb_eccen,5150.0,0.078777,0.152198,0.0000,0.000000,0.0000,0.091000,0.9500
temp_kelvin,4511.0,916.054850,465.218775,34.0000,568.550000,823.0000,1166.500000,4050.0000
star_temp,5772.0,5408.691185,1752.568562,415.0000,4914.750000,5551.0000,5902.000000,57000.0000
star_mass,6035.0,0.940534,0.409806,0.0094,0.776000,0.9400,1.090000,10.9400
system_planet_count,6042.0,1.772261,1.160561,1.0000,1.000000,1.0000,2.000000,8.0000


--------------------------------------------------------------------------
>> Missing values per column
planet_name               0
star_name                 0
planet_mass_earth        31
planet_radius_earth      24
planet_orb_distance     304
planet_orb_eccen        892
temp_kelvin            1531
star_temp               270
star_mass                 7
system_planet_count       0
dtype: int64
--------------------------------------------------------------------------


# EDA

In [5]:
import plotly.express as px

fig = px.histogram(
    initial_df,
    x='planet_mass_earth',
    nbins=50,
    title="Distribution of Exoplanet Masses",
    labels={'planet_mass_earth': 'Planet Mass (Earth Units)'}
)

fig.update_layout(template='plotly_dark')
fig.show()


In [6]:
fig = px.scatter(
    initial_df,
    x='planet_orb_distance',
    y='planet_mass_earth',
    title="Orbital Distance vs Planet Mass",
    labels={
        'planet_orb_distance': 'Orbital Distance (AU)',
        'planet_mass_earth': 'Planet Mass (Earth Units)'
    },
    opacity=0.6
)

fig.update_layout(template='plotly_dark')
fig.show()

#The exoplanet population is dominated by low-mass and gas-giant planets, with Earth-like masses representing a very small fraction of the dataset

### Exploratory Data Understanding

The mass distribution of exoplanets is heavily right-skewed, with the
dataset dominated by gas giants and very few Earth-mass planets.
This highlights the rarity of potentially habitable planets and
motivates the use of imbalance-aware learning techniques.

A scatter analysis of orbital distance versus planetary mass reveals
no simple linear relationship, indicating that orbital stability is
governed by inter-planet spacing and gravitational interactions rather
than mass alone. These observations support the use of physics-informed
features and non-linear classification models.


# Model 1

In [7]:
print("--------------------------------------------------------------------------")
print(">> Fetching dataframe for Model 1 (Stability)...")

stable_columns = [
    'planet_name',
    'star_name',
    'planet_mass_earth',
    'planet_orb_distance',
    'planet_orb_eccen',
    'star_temp',
    'star_mass',
    'system_planet_count'
]

stable_df = initial_df[stable_columns].copy()
print(">> Dataframe fetched successfully")
print("--------------------------------------------------------------------------")

print(f">> Total rows: {len(stable_df)}")
print("--------------------------------------------------------------------------")

print(">> Missing values before cleaning:")
print(stable_df.isnull().sum())
print("--------------------------------------------------------------------------")

print(">> Dropping rows with missing critical values (mass, distance, star mass)...")
stable_df = stable_df.dropna(
    subset=['planet_mass_earth', 'planet_orb_distance', 'star_mass']
)
print(">> Drop completed")
print("--------------------------------------------------------------------------")

print(">> Filling missing orbital eccentricity values with 0...")
stable_df['planet_orb_eccen'] = stable_df['planet_orb_eccen'].fillna(0)
print(">> Fill completed")
print("--------------------------------------------------------------------------")

print(">> Missing values after cleaning:")
print(stable_df.isnull().sum())
print("--------------------------------------------------------------------------")

print(f">> Remaining rows: {len(stable_df)}")
print("--------------------------------------------------------------------------")


--------------------------------------------------------------------------
>> Fetching dataframe for Model 1 (Stability)...
>> Dataframe fetched successfully
--------------------------------------------------------------------------
>> Total rows: 6042
--------------------------------------------------------------------------
>> Missing values before cleaning:
planet_name              0
star_name                0
planet_mass_earth       31
planet_orb_distance    304
planet_orb_eccen       892
star_temp              270
star_mass                7
system_planet_count      0
dtype: int64
--------------------------------------------------------------------------
>> Dropping rows with missing critical values (mass, distance, star mass)...
>> Drop completed
--------------------------------------------------------------------------
>> Filling missing orbital eccentricity values with 0...
>> Fill completed
--------------------------------------------------------------------------
>> Missing va

In [8]:
# ----------------------------------------------------------
# Stability Calculation + Synthetic Unstable Data Generation
# ----------------------------------------------------------

def calculate_stability(input_df):
    """
    Computes Hill-radius-based stability metrics
    for multi-planet systems.
    """
    df = input_df.copy()

    # Neighbor planet parameters (same star)
    df['neighbor_mass_earth'] = (
        df.groupby('star_name')['planet_mass_earth'].shift(1)
    )
    df['neighbor_distance'] = (
        df.groupby('star_name')['planet_orb_distance'].shift(1)
    )

    # Unit conversion: Earth mass → Solar mass
    EARTH_TO_SOLAR = 332_946
    df['mass_solar'] = df['planet_mass_earth'] / EARTH_TO_SOLAR
    df['neighbor_mass_solar'] = df['neighbor_mass_earth'] / EARTH_TO_SOLAR

    # Mutual Hill Radius calculation
    mass_ratio = (
        (df['mass_solar'] + df['neighbor_mass_solar']) /
        (3 * df['star_mass'])
    )

    df['hill_radius'] = (
        mass_ratio ** (1 / 3)
    ) * (
        (df['planet_orb_distance'] + df['neighbor_distance']) / 2
    )

    # Stability score
    df['actual_gap'] = (
        df['planet_orb_distance'] - df['neighbor_distance']
    )
    df['stability_score'] = df['actual_gap'] / df['hill_radius']

    # Drop first planet of each system (no neighbor)
    df = df.dropna(subset=['stability_score']).copy()

    return df


print("--------------------------------------------------------------------------")
print(">> Preparing stability dataset...")

# Keep only multi-planet systems
stable_df = stable_df[stable_df['system_planet_count'] >= 2].copy()

# Sort for correct neighbor assignment
stable_df.sort_values(
    by=['star_name', 'planet_orb_distance'],
    ascending=[True, True],
    inplace=True
)

# Calculate stability metrics
stable_df = calculate_stability(stable_df)
stable_df['is_stable'] = 1

print(f">> Stable systems prepared: {len(stable_df)}")
print("--------------------------------------------------------------------------")

print(">> Generating synthetic unstable systems...")

# Synthetic unstable systems ("Evil Twins")
unstable_df = stable_df.copy()

random_dangerous_gap = np.random.uniform(
    low=0.0,
    high=1.5,
    size=len(unstable_df)
)

unstable_df['planet_orb_distance'] = (
    unstable_df['neighbor_distance'] +
    random_dangerous_gap * unstable_df['hill_radius']
)

unstable_df['actual_gap'] = (
    unstable_df['planet_orb_distance'] -
    unstable_df['neighbor_distance']
)
unstable_df['stability_score'] = (
    unstable_df['actual_gap'] /
    unstable_df['hill_radius']
)

unstable_df['is_stable'] = 0

print(f">> Unstable systems generated: {len(unstable_df)}")
print("--------------------------------------------------------------------------")

# Combine & shuffle dataset
final_stable_df = pd.concat(
    [stable_df, unstable_df],
    ignore_index=True
)

final_stable_df = final_stable_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print(">> Final stability dataset ready")
print(f">> Total samples: {len(final_stable_df)}")
print("--------------------------------------------------------------------------")


--------------------------------------------------------------------------
>> Preparing stability dataset...
>> Stable systems prepared: 1421
--------------------------------------------------------------------------
>> Generating synthetic unstable systems...
>> Unstable systems generated: 1421
--------------------------------------------------------------------------
>> Final stability dataset ready
>> Total samples: 2842
--------------------------------------------------------------------------


In [9]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report,confusion_matrix

print(">> STARTING STABILITY MODEL TRAINING (DECISION TREE)")
print("--------------------------------------------------------------------------")

# Feature selection
feature_cols = [
    'mass_solar',
    'neighbor_mass_solar',
    'planet_orb_distance',
    'neighbor_distance',
    'star_mass',
    'planet_orb_eccen'
]

X = final_stable_df[feature_cols]
y = final_stable_df['is_stable']

print(f">> Features selected : {len(feature_cols)}")
print(f">> Total samples     : {len(X)}")
print("--------------------------------------------------------------------------")

# Train-test split
print(">> Splitting data (80:20)")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("--------------------------------------------------------------------------")

# Model training
print(">> Training Decision Tree model...")
model_dt = DecisionTreeClassifier(random_state=42)
model_dt.fit(X_train, y_train)
print(">> Training completed")
print("--------------------------------------------------------------------------")

# Model evaluation
print(">> MODEL EVALUATION")

y_pred = model_dt.predict(X_test)
accuracy = accuracy_score(y_test, y_pred) * 100

print("--------------------------------------------------------------------------")
print(f"Accuracy: {accuracy:.2f}%")

print("--------------------------------------------------------------------------")
print("Classification Report:")
print(classification_report(y_test, y_pred, digits=2))

print("--------------------------------------------------------------------------")
print("STABILITY MODEL TRAINING COMPLETED")
print("--------------------------------------------------------------------------")


>> STARTING STABILITY MODEL TRAINING (DECISION TREE)
--------------------------------------------------------------------------
>> Features selected : 6
>> Total samples     : 2842
--------------------------------------------------------------------------
>> Splitting data (80:20)
--------------------------------------------------------------------------
>> Training Decision Tree model...
>> Training completed
--------------------------------------------------------------------------
>> MODEL EVALUATION
--------------------------------------------------------------------------
Accuracy: 92.44%
--------------------------------------------------------------------------
Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.92      0.92       283
           1       0.92      0.93      0.92       286

    accuracy                           0.92       569
   macro avg       0.92      0.92      0.92       569
weighted avg       0.92      

In [10]:
import numpy as np
import plotly.express as px

# Confusion matrix values (already computed earlier)
cm = confusion_matrix(y_test, y_pred)

fig = px.imshow(
    cm,
    text_auto=True,
    color_continuous_scale="ice",
    labels=dict(
        x="Predicted Class",
        y="Actual Class",
        color="Number of Systems"
    ),
    x=["Unstable", "Stable"],
    y=["Unstable", "Stable"],
    title="Orbital Stability Classification — Confusion Matrix"
)

fig.update_layout(
    template="plotly_dark",
    font=dict(size=14)
)

fig.show()


#The confusion matrix shows balanced performance across stable and unstable systems, 
# indicating that the model does not favor one class over the other — a critical requirement given the synthetic instability generation.

In [11]:
import pandas as pd
import plotly.express as px

# Extract feature importance from trained Decision Tree
importance_df = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": model_dt.feature_importances_
}).sort_values(by="Importance", ascending=True)

fig = px.bar(
    importance_df,
    x="Importance",
    y="Feature",
    orientation="h",
    color="Importance",
    color_continuous_scale="viridis",
    title="🌌 Feature Importance — Physical Drivers of Orbital Stability"
)

fig.update_layout(
    template="plotly_dark",
    font=dict(size=14)
)

fig.show()

#Orbital distance and inter-planet spacing dominate stability prediction, which aligns with gravitational dynamics and Hill radius theory, 
# validating the physical consistency of the model.

# Model 2

In [12]:
# ----------------------------------------------------------
# Model 2: Habitability Dataset Preparation
# ----------------------------------------------------------

habitable_columns = [
    'planet_name',
    'planet_mass_earth',
    'planet_radius_earth',
    'temp_kelvin'
]

habitable_df = initial_df[habitable_columns].copy()

print("--------------------------------------------------------------------------")
print(">> Preparing habitability dataset")

print(f">> Initial rows: {len(habitable_df)}")
print("\n>> Missing values before cleaning:")
print(habitable_df.isnull().sum())

# Drop rows with critical missing values
habitable_df.dropna(
    subset=['planet_mass_earth', 'planet_radius_earth', 'temp_kelvin'],
    inplace=True
)

print("\n>> Cleaning completed")
print(f">> Rows after cleaning: {len(habitable_df)}")
print("\n>> Missing values after cleaning:")
print(habitable_df.isnull().sum())
print("--------------------------------------------------------------------------")


--------------------------------------------------------------------------
>> Preparing habitability dataset
>> Initial rows: 6042

>> Missing values before cleaning:
planet_name               0
planet_mass_earth        31
planet_radius_earth      24
temp_kelvin            1531
dtype: int64

>> Cleaning completed
>> Rows after cleaning: 4483

>> Missing values after cleaning:
planet_name            0
planet_mass_earth      0
planet_radius_earth    0
temp_kelvin            0
dtype: int64
--------------------------------------------------------------------------


In [13]:
# ----------------------------------------------------------
# Habitability Feature Engineering & ESI Calculation
# ----------------------------------------------------------

print("--------------------------------------------------------------------------")
print(">> Computing habitability-related features...")

# --------------------------------------------------
# Planetary Density (Earth-normalized)
# --------------------------------------------------
habitable_df['density'] = (
    habitable_df['planet_mass_earth'] /
    (habitable_df['planet_radius_earth'] ** 3)
)

# --------------------------------------------------
# Escape Velocity (normalized proxy)
# --------------------------------------------------
habitable_df['escape_velocity'] = np.sqrt(
    habitable_df['planet_mass_earth'] /
    habitable_df['planet_radius_earth']
)

print(">> Density and escape velocity calculated")
print("--------------------------------------------------------------------------")

# --------------------------------------------------
# Earth Similarity Index (ESI) Calculation
# --------------------------------------------------
def calculate_esi(row):
    """
    Computes the Earth Similarity Index (ESI)
    based on radius, density, escape velocity,
    and equilibrium temperature.
    """

    # Earth reference values
    ref_r = 1.0
    ref_d = 1.0
    ref_v = 1.0
    ref_t = 288.0

    # Weight factors
    w_r = 0.57
    w_d = 1.07
    w_v = 0.70
    w_t = 5.58

    # Individual similarity terms
    x_r = (1 - abs((row['planet_radius_earth'] - ref_r) /
                   (row['planet_radius_earth'] + ref_r))) ** w_r

    x_d = (1 - abs((row['density'] - ref_d) /
                   (row['density'] + ref_d))) ** w_d

    x_v = (1 - abs((row['escape_velocity'] - ref_v) /
                   (row['escape_velocity'] + ref_v))) ** w_v

    x_t = (1 - abs((row['temp_kelvin'] - ref_t) /
                   (row['temp_kelvin'] + ref_t))) ** w_t

    # Geometric mean
    return (x_r * x_d * x_v * x_t) ** (1 / 4)


# Apply ESI calculation
habitable_df['ESI'] = habitable_df.apply(calculate_esi, axis=1)

# --------------------------------------------------
# Habitability Label
# --------------------------------------------------
habitable_df['is_habitable'] = habitable_df['ESI'].apply(
    lambda x: 1 if x > 0.8 else 0
)

print(">> ESI calculation completed")
print(f">> Total habitable planets identified: {habitable_df['is_habitable'].sum()}")
print("--------------------------------------------------------------------------")


--------------------------------------------------------------------------
>> Computing habitability-related features...
>> Density and escape velocity calculated
--------------------------------------------------------------------------
>> ESI calculation completed
>> Total habitable planets identified: 53
--------------------------------------------------------------------------


In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import RandomOverSampler

print("--------------------------------------------------------------------------")
print(">> TRAINING HABITABILITY MODEL (LOGISTIC REGRESSION)")
print("--------------------------------------------------------------------------")

# Feature selection
features = ['density', 'escape_velocity', 'temp_kelvin']
X = habitable_df[features]
y = habitable_df['is_habitable']

print(f">> Features used : {features}")
print(f">> Total samples : {len(X)}")
print("--------------------------------------------------------------------------")

# Train-test split (80:20)
print(">> Splitting data (80:20)")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("--------------------------------------------------------------------------")

# Oversampling (training data only)
print(">> Applying Random Oversampling (ROS)")
ros = RandomOverSampler(random_state=42)
X_train_res, y_train_res = ros.fit_resample(X_train, y_train)

print(f">> Training samples (original)  : {len(X_train)}")
print(f">> Training samples (resampled) : {len(X_train_res)}")
print("--------------------------------------------------------------------------")

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_res)
X_test_scaled = scaler.transform(X_test)

# Model training
print(">> Training Logistic Regression model...")
log_model_ros = LogisticRegression()
log_model_ros.fit(X_train_scaled, y_train_res)
print(">> Training completed")
print("--------------------------------------------------------------------------")

# Model evaluation
y_pred = log_model_ros.predict(X_test_scaled)

print(">> MODEL PERFORMANCE")
print("--------------------------------------------------------------------------")
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=2))
print("--------------------------------------------------------------------------")


--------------------------------------------------------------------------
>> TRAINING HABITABILITY MODEL (LOGISTIC REGRESSION)
--------------------------------------------------------------------------
>> Features used : ['density', 'escape_velocity', 'temp_kelvin']
>> Total samples : 4483
--------------------------------------------------------------------------
>> Splitting data (80:20)
--------------------------------------------------------------------------
>> Applying Random Oversampling (ROS)
>> Training samples (original)  : 3586
>> Training samples (resampled) : 7084
--------------------------------------------------------------------------
>> Training Logistic Regression model...
>> Training completed
--------------------------------------------------------------------------
>> MODEL PERFORMANCE
--------------------------------------------------------------------------
Accuracy: 92.87%

Classification Report:
              precision    recall  f1-score   support

           

In [15]:
import numpy as np
import plotly.express as px

cm = confusion_matrix(y_test, y_pred)

fig = px.imshow(
    cm,
    text_auto=True,
    color_continuous_scale="plasma",
    labels=dict(
        x="Predicted Class",
        y="Actual Class",
        color="Number of Planets"
    ),
    x=["Not Habitable", "Habitable"],
    y=["Not Habitable", "Habitable"],
    title="Habitability Classification — Confusion Matrix"
)

fig.update_layout(
    template="plotly_dark",
    font=dict(size=14)
)

fig.show()

#The confusion matrix demonstrates that the model successfully identifies rare habitable planets, confirming that oversampling prevented majority-class dominance.

In [16]:
import pandas as pd
import plotly.express as px

coef_df = pd.DataFrame({
    "Feature": features,
    "Coefficient": log_model_ros.coef_[0]
}).sort_values(by="Coefficient")

fig = px.bar(
    coef_df,
    x="Coefficient",
    y="Feature",
    orientation="h",
    color="Coefficient",
    color_continuous_scale="viridis",
    title="Physical Drivers of Planetary Habitability (Logistic Regression)"
)

fig.update_layout(
    template="plotly_dark",
    font=dict(size=14)
)

fig.show()

# Positive coefficients indicate physical properties that increase habitability probability, while negative coefficients suppress it, 
# making the model directly interpretable in terms of planetary physics.